## Exercise 2 ##

### Sound1.wav ###

In [95]:
import wave
import struct
import os
import numpy as np
from bitarray import bitarray

# Read WAV file
def read_wav_file(filename):
    with wave.open(filename, 'rb') as wav_file:
        num_channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()
        frame_rate = wav_file.getframerate()
        num_frames = wav_file.getnframes()
        frames = wav_file.readframes(num_frames)
    samples = np.frombuffer(frames, dtype=np.int16)
    return samples, num_channels, frame_rate, sample_width

# Second-order predictor
def compute_residuals(samples):
    residuals = np.zeros_like(samples, dtype=np.int32)
    residuals[0] = samples[0]
    if len(samples) > 1:
        residuals[1] = samples[1] - samples[0]
    for i in range(2, len(samples)):
        predicted = 2 * samples[i-1] - samples[i-2]
        residuals[i] = samples[i] - predicted
    return residuals

# Simplified mapping: absolute value + sign bit
def encode_residuals(residuals):
    abs_residuals = np.abs(residuals).astype(np.uint32)
    signs = (residuals < 0).astype(np.uint8)  # 1 for negative, 0 for positive
    return abs_residuals, signs

# Rice encoding
def rice_encode(u, k):
    q = u >> k
    r = u & ((1 << k) - 1)
    unary = bitarray('0' * q + '1')
    binary = bitarray(format(r, f'0{k}b'))
    return unary + binary

# Encode to file with sign bits
def encode_to_file(samples, k, output_filename):
    residuals = compute_residuals(samples)
    abs_residuals, signs = encode_residuals(residuals)
    num_samples = len(residuals)
    
    bit_stream = bitarray()
    for i in range(num_samples):
        bit_stream.append(signs[i])  # Add sign bit
        bit_stream.extend(rice_encode(abs_residuals[i], k))  # Encode magnitude
    
    if len(bit_stream) % 8:
        bit_stream.extend('0' * (8 - len(bit_stream) % 8))
    
    header = struct.pack('>I B', num_samples, k)
    with open(output_filename, 'wb') as f:
        f.write(header)
        f.write(bit_stream.tobytes())
    
    return len(header) + len(bit_stream.tobytes())

# Decode from file
def decode_from_file(filename):
    with open(filename, 'rb') as f:
        header = f.read(5)
        num_samples, k = struct.unpack('>I B', header)
        bit_data = f.read()
        bits = bitarray()
        bits.frombytes(bit_data)
    
    abs_residuals = []
    signs = []
    pos = 0
    bits_str = bits.to01()
    while pos < len(bits_str) and len(abs_residuals) < num_samples:
        # Read sign bit
        sign = int(bits_str[pos])
        signs.append(sign)
        pos += 1
        
        # Read Rice-coded magnitude
        q = 0
        while pos < len(bits_str) and bits_str[pos] == '0':
            q += 1
            pos += 1
        if pos >= len(bits_str):
            break
        pos += 1  # Skip '1'
        if pos + k > len(bits_str):
            break
        r = int(bits_str[pos:pos + k], 2)
        pos += k
        u = (q << k) + r
        abs_residuals.append(u)
    
    # Reconstruct residuals with signs
    residuals = np.where(np.array(signs) == 0, np.array(abs_residuals), -np.array(abs_residuals)).astype(np.int32)
    
    # Reconstruct samples
    samples = np.zeros_like(residuals, dtype=np.int16)
    samples[0] = residuals[0]
    if len(residuals) > 1:
        samples[1] = residuals[1] + samples[0]
    for i in range(2, len(residuals)):
        predicted = 2 * samples[i-1] - samples[i-2]
        samples[i] = predicted + residuals[i]
    
    return samples[:num_samples]

# Write WAV file
def write_wav_file(samples, num_channels, frame_rate, sample_width, output_filename):
    with wave.open(output_filename, 'wb') as wav_file:
        wav_file.setnchannels(num_channels)
        wav_file.setsampwidth(sample_width)
        wav_file.setframerate(frame_rate)
        wav_file.writeframes(samples.tobytes())

# Process audio file
def process_audio_file(input_filename, k_values=[2, 4]):
    samples, num_channels, frame_rate, sample_width = read_wav_file(input_filename)
    original_size = os.path.getsize(input_filename)
    results = {'Original Size': original_size}
    
    for k in k_values:
        encoded_filename = input_filename.replace('.wav', f'_Enc{k}.ex2')
        compressed_size = encode_to_file(samples, k, encoded_filename)
        results[f'Rice (K={k} bits)'] = compressed_size
        
        decoded_samples = decode_from_file(encoded_filename)
        decoded_filename = input_filename.replace('.wav', f'_Enc_Dec{k}.wav')
        write_wav_file(decoded_samples, num_channels, frame_rate, sample_width, decoded_filename)
        
        if np.array_equal(decoded_samples, samples):
            print(f"Lossless compression verified for {input_filename} with k={k}")
        else:
            print(f"Warning: Lossless compression failed for {input_filename} with k={k}")
            print(f"First 10 original samples: {samples[:10]}")
            print(f"First 10 decoded samples: {decoded_samples[:10]}")
    
    for k in k_values:
        compressed_size = results[f'Rice (K={k} bits)']
        percent_compression = ((original_size - compressed_size) / original_size) * 100
        results[f'% Compression (K={k} bits)'] = percent_compression
    
    return results

# Example usage
input_files = ['Sound1.wav']
k_values = [2, 4]
table_data = {}

for input_file in input_files:
    if os.path.exists(input_file):
        results = process_audio_file(input_file, k_values)
        table_data[input_file] = results
    else:
        print(f"File {input_file} not found.")

# Print table
print("\nCompression Analysis Table:")
print("| Audio File  | Original Size | Rice (K=4 bits) | Rice (K=2 bits) | % Compression (K=4 bits) | % Compression (K=2 bits) |")
print("|-------------|---------------|-----------------|-----------------|--------------------------|--------------------------|")
for file, data in table_data.items():
    print(f"| {file:<11} | {data['Original Size']:>13} | {data['Rice (K=4 bits)']:>15} | {data['Rice (K=2 bits)']:>15} | {data['% Compression (K=4 bits)']:>23.2f} | {data['% Compression (K=2 bits)']:>23.2f} |")

Lossless compression verified for Sound1.wav with k=2
Lossless compression verified for Sound1.wav with k=4

Compression Analysis Table:
| Audio File  | Original Size | Rice (K=4 bits) | Rice (K=2 bits) | % Compression (K=4 bits) | % Compression (K=2 bits) |
|-------------|---------------|-----------------|-----------------|--------------------------|--------------------------|
| Sound1.wav  |       1002088 |          565193 |         1063174 |                   43.60 |                   -6.10 |


### Sound2.wav ###

In [97]:
import wave
import struct
import os
import numpy as np
from bitarray import bitarray

# Read WAV file
def read_wav_file(filename):
    with wave.open(filename, 'rb') as wav_file:
        num_channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()
        frame_rate = wav_file.getframerate()
        num_frames = wav_file.getnframes()
        frames = wav_file.readframes(num_frames)
    samples = np.frombuffer(frames, dtype=np.int16)
    return samples, num_channels, frame_rate, sample_width

# Second-order predictor
def compute_residuals(samples):
    residuals = np.zeros_like(samples, dtype=np.int32)
    residuals[0] = samples[0]
    if len(samples) > 1:
        residuals[1] = samples[1] - samples[0]
    for i in range(2, len(samples)):
        predicted = 2 * samples[i-1] - samples[i-2]
        residuals[i] = samples[i] - predicted
    return residuals

# Simplified mapping: absolute value + sign bit
def encode_residuals(residuals):
    abs_residuals = np.abs(residuals).astype(np.uint32)
    signs = (residuals < 0).astype(np.uint8)  # 1 for negative, 0 for positive
    return abs_residuals, signs

# Rice encoding
def rice_encode(u, k):
    q = u >> k
    r = u & ((1 << k) - 1)
    unary = bitarray('0' * q + '1')
    binary = bitarray(format(r, f'0{k}b'))
    return unary + binary

# Encode to file with sign bits
def encode_to_file(samples, k, output_filename):
    residuals = compute_residuals(samples)
    abs_residuals, signs = encode_residuals(residuals)
    num_samples = len(residuals)
    
    bit_stream = bitarray()
    for i in range(num_samples):
        bit_stream.append(signs[i])  # Add sign bit
        bit_stream.extend(rice_encode(abs_residuals[i], k))  # Encode magnitude
    
    if len(bit_stream) % 8:
        bit_stream.extend('0' * (8 - len(bit_stream) % 8))
    
    header = struct.pack('>I B', num_samples, k)
    with open(output_filename, 'wb') as f:
        f.write(header)
        f.write(bit_stream.tobytes())
    
    return len(header) + len(bit_stream.tobytes())

# Decode from file
def decode_from_file(filename):
    with open(filename, 'rb') as f:
        header = f.read(5)
        num_samples, k = struct.unpack('>I B', header)
        bit_data = f.read()
        bits = bitarray()
        bits.frombytes(bit_data)
    
    abs_residuals = []
    signs = []
    pos = 0
    bits_str = bits.to01()
    while pos < len(bits_str) and len(abs_residuals) < num_samples:
        # Read sign bit
        sign = int(bits_str[pos])
        signs.append(sign)
        pos += 1
        
        # Read Rice-coded magnitude
        q = 0
        while pos < len(bits_str) and bits_str[pos] == '0':
            q += 1
            pos += 1
        if pos >= len(bits_str):
            break
        pos += 1  # Skip '1'
        if pos + k > len(bits_str):
            break
        r = int(bits_str[pos:pos + k], 2)
        pos += k
        u = (q << k) + r
        abs_residuals.append(u)
    
    # Reconstruct residuals with signs
    residuals = np.where(np.array(signs) == 0, np.array(abs_residuals), -np.array(abs_residuals)).astype(np.int32)
    
    # Reconstruct samples
    samples = np.zeros_like(residuals, dtype=np.int16)
    samples[0] = residuals[0]
    if len(residuals) > 1:
        samples[1] = residuals[1] + samples[0]
    for i in range(2, len(residuals)):
        predicted = 2 * samples[i-1] - samples[i-2]
        samples[i] = predicted + residuals[i]
    
    return samples[:num_samples]

# Write WAV file
def write_wav_file(samples, num_channels, frame_rate, sample_width, output_filename):
    with wave.open(output_filename, 'wb') as wav_file:
        wav_file.setnchannels(num_channels)
        wav_file.setsampwidth(sample_width)
        wav_file.setframerate(frame_rate)
        wav_file.writeframes(samples.tobytes())

# Process audio file
def process_audio_file(input_filename, k_values=[2, 4]):
    samples, num_channels, frame_rate, sample_width = read_wav_file(input_filename)
    original_size = os.path.getsize(input_filename)
    results = {'Original Size': original_size}
    
    for k in k_values:
        encoded_filename = input_filename.replace('.wav', f'_Enc{k}.ex2')
        compressed_size = encode_to_file(samples, k, encoded_filename)
        results[f'Rice (K={k} bits)'] = compressed_size
        
        decoded_samples = decode_from_file(encoded_filename)
        decoded_filename = input_filename.replace('.wav', f'_Enc_Dec{k}.wav')
        write_wav_file(decoded_samples, num_channels, frame_rate, sample_width, decoded_filename)
        
        if np.array_equal(decoded_samples, samples):
            print(f"Lossless compression verified for {input_filename} with k={k}")
        else:
            print(f"Warning: Lossless compression failed for {input_filename} with k={k}")
            print(f"First 10 original samples: {samples[:10]}")
            print(f"First 10 decoded samples: {decoded_samples[:10]}")
    
    for k in k_values:
        compressed_size = results[f'Rice (K={k} bits)']
        percent_compression = ((original_size - compressed_size) / original_size) * 100
        results[f'% Compression (K={k} bits)'] = percent_compression
    
    return results

# Example usage
input_files = ['Sound2.wav']
k_values = [2, 4]
table_data = {}

for input_file in input_files:
    if os.path.exists(input_file):
        results = process_audio_file(input_file, k_values)
        table_data[input_file] = results
    else:
        print(f"File {input_file} not found.")

# Print table
print("\nCompression Analysis Table:")
print("| Audio File  | Original Size | Rice (K=4 bits) | Rice (K=2 bits) | % Compression (K=4 bits) | % Compression (K=2 bits) |")
print("|-------------|---------------|-----------------|-----------------|--------------------------|--------------------------|")
for file, data in table_data.items():
    print(f"| {file:<11} | {data['Original Size']:>13} | {data['Rice (K=4 bits)']:>15} | {data['Rice (K=2 bits)']:>15} | {data['% Compression (K=4 bits)']:>23.2f} | {data['% Compression (K=2 bits)']:>23.2f} |")

Lossless compression verified for Sound2.wav with k=2
Lossless compression verified for Sound2.wav with k=4

Compression Analysis Table:
| Audio File  | Original Size | Rice (K=4 bits) | Rice (K=2 bits) | % Compression (K=4 bits) | % Compression (K=2 bits) |
|-------------|---------------|-----------------|-----------------|--------------------------|--------------------------|
| Sound2.wav  |       1008044 |        56546703 |       225021309 |                -5509.55 |               -22222.57 |


## Further Improvement ##

In [116]:
import wave
import struct
import os
import numpy as np
from bitarray import bitarray

# Read WAV file
def read_wav_file(filename):
    with wave.open(filename, 'rb') as wav_file:
        num_channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()
        frame_rate = wav_file.getframerate()
        num_frames = wav_file.getnframes()
        frames = wav_file.readframes(num_frames)
    samples = np.frombuffer(frames, dtype=np.int16)
    return samples, num_channels, frame_rate, sample_width

# Stereo decorrelation
def stereo_decorrelate(samples, num_channels):
    if num_channels == 2:
        left = samples[0::2]
        right = samples[1::2]
        mid = (left + right) // 2
        side = left - right
        return np.vstack((mid, side)).T.flatten()
    return samples

# Reverse stereo decorrelation
def stereo_correlate(samples, num_channels):
    if num_channels == 2:
        mid = samples[0::2]
        side = samples[1::2]
        left = mid + side
        right = mid - side
        return np.vstack((left, right)).T.flatten()
    return samples

# LPC predictor (order 4, vectorized)
def compute_lpc_residuals(samples, order=4):
    residuals = np.zeros_like(samples, dtype=np.int32)
    residuals[:order] = samples[:order]
    coeffs = np.array([0.95, -0.5, 0.3, -0.1])  # Optimized for general audio
    if len(samples) > order:
        past_samples = np.lib.stride_tricks.sliding_window_view(samples, order)[:(len(samples)-order)]
        predicted = np.dot(past_samples, coeffs[::-1]).astype(np.int32)
        residuals[order:] = samples[order:] - predicted
    return residuals

# Reconstruct samples with LPC
def reconstruct_samples(residuals, order=4):
    samples = np.zeros_like(residuals, dtype=np.int16)
    samples[:order] = residuals[:order].astype(np.int16)
    coeffs = np.array([0.95, -0.5, 0.3, -0.1])
    if len(residuals) > order:
        past_samples = np.lib.stride_tricks.sliding_window_view(samples, order)[:(len(samples)-order)]
        predicted = np.dot(past_samples, coeffs[::-1]).astype(np.int32)
        samples[order:] = (predicted + residuals[order:]).astype(np.int16)
    return samples

# Rice coding with adaptive k

def select_rice_k(residuals):
    abs_residuals = np.abs(residuals)
    mean_log2 = np.mean(np.log2(abs_residuals + 1))  # Estimate optimal k
    k = max(0, min(15, int(mean_log2 - 1)))  # Range 0-15
    return k

def rice_encode(residuals, k):
    bit_stream = bitarray()
    for r in residuals:
        sign = 1 if r < 0 else 0
        u = abs(r)
        q = u >> k
        rem = u & ((1 << k) - 1)
        bit_stream.append(sign)
        bit_stream.extend('0' * q + '1')
        # For k==0, this adds an empty string (which is fine)
        bit_stream.extend(format(rem, f'0{k}b'))
    return bit_stream

def rice_decode(bit_stream, k, num_samples):
    residuals = []
    pos = 0
    while pos < len(bit_stream) and len(residuals) < num_samples:
        # Read sign bit
        sign = bit_stream[pos]
        pos += 1
        q = 0
        # Count the number of 0's until a 1 is reached (unary part)
        while pos < len(bit_stream) and bit_stream[pos] == 0:
            q += 1
            pos += 1
        if pos >= len(bit_stream):
            break
        pos += 1  # Skip the '1'
        # If k==0, no bits to read for remainder; set r_val = 0
        if k == 0:
            r_val = 0
        else:
            if pos + k > len(bit_stream):
                break
            r_val = int(bit_stream[pos:pos + k].to01(), 2)
            pos += k
        value = (-1 if sign else 1) * ((q << k) + r_val)
        residuals.append(value)
    return np.array(residuals, dtype=np.int32)

# Encode to file with block processing
def encode_to_file(samples, num_channels, output_filename, block_size=4096):
    preprocessed_samples = stereo_decorrelate(samples, num_channels)
    num_samples = len(preprocessed_samples)
    # Using LPC predictor of order 4
    residuals = compute_lpc_residuals(preprocessed_samples)
    
    bit_stream = bitarray()
    block_headers = []
    for start in range(0, num_samples, block_size):
        end = min(start + block_size, num_samples)
        block_residuals = residuals[start:end]
        k = select_rice_k(block_residuals)
        block_bit_stream = rice_encode(block_residuals, k)
        block_headers.append((k, len(block_bit_stream)))
        bit_stream.extend(block_bit_stream)
    
    if len(bit_stream) % 8:
        bit_stream.extend('0' * (8 - len(bit_stream) % 8))
    
    # Write header: 4 bytes num_samples, 1 byte num_channels, 4 bytes num_blocks
    header = struct.pack('>I B I', num_samples, num_channels, len(block_headers))
    total_size = len(header)
    with open(output_filename, 'wb') as f:
        f.write(header)
        for k, bits_len in block_headers:
            f.write(struct.pack('>B I', k, bits_len))  # k as 1 byte, bits_len as 4 bytes
            total_size += 5
        encoded_bytes = bit_stream.tobytes()
        f.write(encoded_bytes)
        total_size += len(encoded_bytes)
    
    return total_size

# Decode from file
def decode_from_file(filename):
    with open(filename, 'rb') as f:
        header = f.read(9)
        if len(header) < 9:
            raise ValueError(f"File {filename} too short: only {len(header)} bytes")
        num_samples, num_channels, num_blocks = struct.unpack('>I B I', header)
        
        block_headers = []
        for _ in range(num_blocks):
            header_data = f.read(5)
            if len(header_data) < 5:
                raise ValueError("Incomplete block header")
            k, bits_len = struct.unpack('>B I', header_data)
            block_headers.append((k, bits_len))
        
        bit_data = f.read()
        bits = bitarray()
        bits.frombytes(bit_data)
    
    residuals = np.zeros(num_samples, dtype=np.int32)
    bit_pos = 0
    for i, block_start in enumerate(range(0, num_samples, 4096)):
        block_end = min(block_start + 4096, num_samples)
        k, block_bits_len = block_headers[i]
        block_bits = bits[bit_pos:bit_pos + block_bits_len]
        block_residuals = rice_decode(block_bits, k, block_end - block_start)
        residuals[block_start:block_end] = block_residuals
        bit_pos += block_bits_len
    
    reconstructed = reconstruct_samples(residuals)
    # Reverse the stereo decorrelation
    return stereo_correlate(reconstructed, num_channels)

# Write WAV file
def write_wav_file(samples, num_channels, frame_rate, sample_width, output_filename):
    with wave.open(output_filename, 'wb') as wav_file:
        wav_file.setnchannels(num_channels)
        wav_file.setsampwidth(sample_width)
        wav_file.setframerate(frame_rate)
        wav_file.writeframes(samples.tobytes())

# Process audio file
def process_audio_file(input_filename):
    samples, num_channels, frame_rate, sample_width = read_wav_file(input_filename)
    original_size = os.path.getsize(input_filename)
    results = {'Original Size': original_size}
    
    encoded_filename = input_filename.replace('.wav', '_Enc.ex2')
    compressed_size = encode_to_file(samples, num_channels, encoded_filename)
    results['FLAC-like'] = compressed_size
    
    decoded_samples = decode_from_file(encoded_filename)
    decoded_filename = input_filename.replace('.wav', '_Enc_Dec.wav')
    write_wav_file(decoded_samples, num_channels, frame_rate, sample_width, decoded_filename)
    
    if np.array_equal(decoded_samples, samples):
        print(f"Lossless compression verified for {input_filename}")
    else:
        print(f"Warning: Lossless compression failed for {input_filename}")
        print(f"First 10 original samples: {samples[:10]}")
        print(f"First 10 decoded samples: {decoded_samples[:10]}")
        diff = np.where(samples != decoded_samples)[0]
        if len(diff) > 0:
            print(f"Differences at indices: {diff[:10]}")
            print(f"Sample values: {samples[diff[:10]]} vs {decoded_samples[diff[:10]]}")
    
    percent_compression = ((original_size - compressed_size) / original_size) * 100
    results['% Compression'] = percent_compression
    
    return results

# Example usage
input_files = ['Sound1.wav', 'Sound2.wav']
table_data = {}

for input_file in input_files:
    if os.path.exists(input_file):
        results = process_audio_file(input_file)
        table_data[input_file] = results
    else:
        print(f"File {input_file} not found.")

# Print table
print("\nCompression Analysis Table:")
print("| Audio File  | Original Size | FLAC-like (bytes) | % Compression |")
print("|-------------|---------------|-------------------|---------------|")
for file, data in table_data.items():
    print(f"| {file:<11} | {data['Original Size']:>13} | {data['FLAC-like']:>17} | {data['% Compression']:>13.2f} |")

First 10 original samples: [-7 -7 -7 -7 -8 -7 -6 -7 -5 -5]
First 10 decoded samples: [-7 -7 -7 -7 -8  0 -3 -4 -1 -3]
Differences at indices: [ 5  6  7  8  9 10 11 12 13 14]
Sample values: [-7 -6 -7 -5 -5 -6 -5 -6 -7 -5] vs [ 0 -3 -4 -1 -3 -3 -2 -4 -3 -1]
First 10 original samples: [ -999   886 -1325   886 -1514   957 -1861   766 -2008  1067]
First 10 decoded samples: [ -999   886 -1325   886 -1514  2395 -3527  3467 -4104  4011]
Differences at indices: [ 5  6  7  8  9 10 11 12 13 14]
Sample values: [  957 -1861   766 -2008  1067 -2068   884 -2237   725 -2216] vs [ 2395 -3527  3467 -4104  4011 -4501  4061 -4631  4019 -4495]

Compression Analysis Table:
| Audio File  | Original Size | FLAC-like (bytes) | % Compression |
|-------------|---------------|-------------------|---------------|
| Sound1.wav  |       1002088 |            622264 |         37.90 |
| Sound2.wav  |       1008044 |           1115029 |        -10.61 |


### Tested with Huffman ###

In [122]:
import wave
import struct
import os
import numpy as np
from bitarray import bitarray
import heapq

# WAV File
def read_wav_file(filename):
    with wave.open(filename, 'rb') as wav_file:
        num_channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()
        frame_rate = wav_file.getframerate()
        num_frames = wav_file.getnframes()
        frames = wav_file.readframes(num_frames)
    samples = np.frombuffer(frames, dtype=np.int16)
    return samples, num_channels, frame_rate, sample_width

def write_wav_file(samples, num_channels, frame_rate, sample_width, output_filename):
    with wave.open(output_filename, 'wb') as wav_file:
        wav_file.setnchannels(num_channels)
        wav_file.setsampwidth(sample_width)
        wav_file.setframerate(frame_rate)
        wav_file.writeframes(samples.tobytes())

# Predictive Coding (Lossless)
def compute_residuals(samples, order):
    residuals = np.zeros_like(samples, dtype=np.int32)
    residuals[0] = samples[0]
    if len(samples) > 1:
        residuals[1] = samples[1] - samples[0]
    for i in range(2, len(samples)):
        if order == 1:
            predicted = samples[i-1]
        elif order == 2:
            predicted = 2 * samples[i-1] - samples[i-2]
        else:
            # For orders > 2, fallback to order 1
            predicted = samples[i-1]
        residuals[i] = samples[i] - predicted
    return residuals

# Mapping residuals to nonnegative integers (and back)
def map_residual(r):
    return int(2 * r) if r >= 0 else int(-2 * r - 1)

def inverse_map_residual(u):
    return u // 2 if u % 2 == 0 else -(u + 1) // 2

# Huffman Coding Helper Functions
def build_frequency_table(data):
    freq = {}
    for d in data:
        freq[d] = freq.get(d, 0) + 1
    return freq

def build_huffman_tree(freq):
    heap = []
    count = 0
    for symbol, frequency in freq.items():
        heapq.heappush(heap, (frequency, count, symbol))
        count += 1
    while len(heap) > 1:
        freq1, count1, left = heapq.heappop(heap)
        freq2, count2, right = heapq.heappop(heap)
        new_node = (left, right)
        heapq.heappush(heap, (freq1 + freq2, count, new_node))
        count += 1
    return heap[0][2]

def build_huffman_codes(tree, prefix=""):
    codes = {}
    if isinstance(tree, int):
        codes[tree] = prefix if prefix != "" else "0"
    else:
        left, right = tree
        codes.update(build_huffman_codes(left, prefix + "0"))
        codes.update(build_huffman_codes(right, prefix + "1"))
    return codes

def huffman_encode(data, codes):
    encoded = bitarray()
    for d in data:
        encoded.extend(codes[d])
    return encoded

def huffman_decode(encoded, tree, num_symbols):
    decoded = []
    node = tree
    for bit in encoded:
        node = node[0] if bit == 0 else node[1]
        if isinstance(node, int):
            decoded.append(node)
            node = tree
            if len(decoded) == num_symbols:
                break
    return decoded

# File Compression with Huffman Coding
def encode_to_file_huffman(samples, order, output_filename):
    # Compute prediction residuals
    residuals = compute_residuals(samples, order)
    # Map residuals to nonnegative integers
    mapped = [map_residual(r) for r in residuals]
    num_samples = len(mapped)
    
    # Build frequency table and Huffman codes
    freq_table = build_frequency_table(mapped)
    tree = build_huffman_tree(freq_table)
    codes = build_huffman_codes(tree)
    
    # Encode the mapped residuals using Huffman codes
    encoded = huffman_encode(mapped, codes)
    # Ensure bit stream is byte-aligned
    if len(encoded) % 8:
        encoded.extend('0' * (8 - len(encoded) % 8))
    encoded_bytes = encoded.tobytes()
    
    header = struct.pack('>I B', num_samples, order)
    header += struct.pack('>I', len(freq_table))
    # For each entry: 4 bytes symbol, 4 bytes frequency
    for symbol, frequency in freq_table.items():
        header += struct.pack('>I I', symbol, frequency)
    
    with open(output_filename, 'wb') as f:
        f.write(header)
        f.write(encoded_bytes)
    
    return len(header) + len(encoded_bytes)

def decode_from_file_huffman(filename):
    with open(filename, 'rb') as f:
        # Read basic header: number of samples and predictor order
        header = f.read(5)
        num_samples, order = struct.unpack('>I B', header)
        # Read frequency table size
        table_size_bytes = f.read(4)
        num_entries = struct.unpack('>I', table_size_bytes)[0]
        freq_table = {}
        for _ in range(num_entries):
            entry = f.read(8)
            symbol, frequency = struct.unpack('>I I', entry)
            freq_table[symbol] = frequency
        # Rebuild Huffman tree
        tree = build_huffman_tree(freq_table)
        # Read the rest of the file as the encoded bit stream
        encoded_bytes = f.read()
        encoded = bitarray()
        encoded.frombytes(encoded_bytes)
    
    # Decode the bit stream
    mapped = huffman_decode(encoded, tree, num_samples)
    # Inverse map to get residuals
    residuals = [inverse_map_residual(u) for u in mapped]
    
    # Reconstruct the original samples from residuals
    samples = np.zeros(num_samples, dtype=np.int16)
    samples[0] = residuals[0]
    if num_samples > 1:
        samples[1] = residuals[1] + samples[0]
    for i in range(2, num_samples):
        if order == 1:
            predicted = samples[i-1]
        elif order == 2:
            predicted = 2 * samples[i-1] - samples[i-2]
        else:
            predicted = samples[i-1]
        samples[i] = predicted + residuals[i]
    return samples

# Processing the Audio File with Huffman Compression
def process_audio_file_huffman(input_filename, order_values=[1, 2]):
    samples, num_channels, frame_rate, sample_width = read_wav_file(input_filename)
    original_size = os.path.getsize(input_filename)
    results = {'Original Size': original_size}
    
    for order in order_values:
        encoded_filename = input_filename.replace('.wav', f'_HuffOrder{order}.bin')
        compressed_size = encode_to_file_huffman(samples, order, encoded_filename)
        results[f'Huffman (Order={order})'] = compressed_size
        
        decoded_samples = decode_from_file_huffman(encoded_filename)
        decoded_filename = input_filename.replace('.wav', f'_HuffOrder{order}_Dec.wav')
        write_wav_file(decoded_samples, num_channels, frame_rate, sample_width, decoded_filename)
        
        if np.array_equal(decoded_samples, samples):
            print(f"Lossless compression verified for {input_filename} with predictor order={order}")
        else:
            print(f"Warning: Lossless compression failed for {input_filename} with predictor order={order}")
    
    for order in order_values:
        compressed_size = results[f'Huffman (Order={order})']
        percent_compression = ((original_size - compressed_size) / original_size) * 100
        results[f'% Compression (Order={order})'] = percent_compression
    
    return results

#Implementation
if __name__ == "__main__":
    input_files = ['Sound1.wav', 'Sound2.wav']
    order_values = [1, 2, 3]
    table_data = {}

    for input_file in input_files:
        if os.path.exists(input_file):
            results = process_audio_file_huffman(input_file, order_values)
            table_data[input_file] = results
        else:
            print(f"File {input_file} not found.")

    # Print summary table
    print("\nCompression Analysis Table:")
    print("| Audio File  | Original Size | Huffman (Order=2) | Huffman (Order=1) | % Compression (Order=2) | % Compression (Order=1) |")
    print("|-------------|---------------|-------------------|-------------------|-------------------------|-------------------------|")
    for file, data in table_data.items():
        print(f"| {file:<11} | {data['Original Size']:>13} | {data['Huffman (Order=2)']:>17} | {data['Huffman (Order=1)']:>17} | {data['% Compression (Order=2)']:>23.2f} | {data['% Compression (Order=1)']:>23.2f} |")

Lossless compression verified for Sound1.wav with predictor order=1
Lossless compression verified for Sound1.wav with predictor order=2
Lossless compression verified for Sound1.wav with predictor order=3


/var/folders/2f/1m_zjpz92ds3p9_6hy8t_b8h0000gn/T/ipykernel_16361/2852428061.py:40: RuntimeWarning: overflow encountered in scalar subtract
  residuals[i] = samples[i] - predicted


Lossless compression verified for Sound2.wav with predictor order=1
Lossless compression verified for Sound2.wav with predictor order=2
Lossless compression verified for Sound2.wav with predictor order=3

Compression Analysis Table:
| Audio File  | Original Size | Huffman (Order=2) | Huffman (Order=1) | % Compression (Order=2) | % Compression (Order=1) |
|-------------|---------------|-------------------|-------------------|-------------------------|-------------------------|
| Sound1.wav  |       1002088 |            466967 |            500923 |                   53.40 |                   50.01 |
| Sound2.wav  |       1008044 |           1767009 |           1389802 |                  -75.29 |                  -37.87 |
